# Accelerated Computing with JAX including AD

In this notebook, we will:
1. Review basic JAX concepts
2. Demonstrate AD functionality, JIT, and automatic vectorization
3. Optimization using Optax
4. Probabilistic programming with NumPyro


JAX provides a NumPy-compatible API via jax.numpy, usually imported as jnp.

At first glance, JAX arrays behave like NumPy arrays.

In [ ]:
import jax
import jax.numpy as jnp

x = jnp.array([1.0, 2.0, 3.0])
print(x)

JAX arrays are immutable.

This makes programs easier to analyze, differentiate, and compile — but it requires a small shift in mindset.

In [ ]:
x = jnp.array([1.0, 2.0, 3.0])
x[0] = 10.0   # Uncommenting this will raise an error

- No in-place writes
- Prevents hidden side effects
- Enables safe transformations like jit and vmap

But JAX arrays have a work around:

In [ ]:
x_new = x.at[0].set(10.0)
print(f"x = {x}, x_new = {x_new}")

## JAX API

The most useful JAX libraries for scientific computing are:

- `jax.numpy` is a drop-in replacement for NumPy.
- `jax.scipy` is a drop-in replacement for SciPy.

Other JAX libraries that you might find usefule:

- `jax.random` for pseudo-random number generation
- `jax.lax` for lower-level operations and control flow

## JAX Automatic Differentiation (AD)

JAX computes derivatives by transforming functions.
`jax.grad(f)` returns a new function that evaluates ∂f/∂x.
- No .backward()
- No graph objects (no tracking of tensors and operations)
- Differentiation is composable (grad-of-grad, jit-of-grad, vmap-of-grad, etc.)

In [ ]:
import jax
import jax.numpy as jnp

# Our function
def f(x):
    return jnp.sin(x) + x**2

print(jax.make_jaxpr(f)(2.0))

In [ ]:

# Compute the derivative of f
dfdx = jax.grad(f)

print("∂f(2.0)/∂x  =", dfdx(2.0))

By default, grad differentiates w.r.t. the first argument.
Use argnums to choose which argument(s) to differentiate.

In [ ]:
def g(a, b):
    return a**2 + 3*b + a*b

dg_da = jax.grad(g, argnums=0)
dg_db = jax.grad(g, argnums=1)

a, b = 2.0, 5.0
print("∂g/∂a =", dg_da(a, b))
print("∂g/∂b =", dg_db(a, b))

You can also get graidents of all arguments as a returned set.

In [ ]:
dg_dab = jax.grad(g, argnums=(0, 1))
print("∂g/∂(a,b) =", dg_dab(a, b))

In optimization/inference you often want both the value and the gradient.
`jax.value_and_grad` gives you both without recomputing work (when jitted/traced).

In [ ]:
# Define a mean squared error loss function
def mse_loss(theta, x, y):
    yhat = theta[0] * x + theta[1]
    return jnp.mean((yhat - y)**2)

# Create some data
x = jnp.linspace(0.0, 1.0, 100)
y = 2.0 * x + 0.1

# Create an array of parameters (slope and intercept)
theta = jnp.array([1.0, 0.0])

# Get both loss value and gradient
loss_and_grad = jax.value_and_grad(mse_loss)

# Evaluate at theta given x and y, note theta now is an array so grad is also an array
val, grad = loss_and_grad(theta, x, y)
print("loss =", val)
print("grad =", grad)

## Jacobians

`jax.grad` requires the function output to be a scalar.

If you have vector output, you often:
- reduce it to a scalar (sum/mean), or
- use Jacobians (`jax.jacfwd`, `jax.jacrev`).



A Jacobian is “all partial derivatives of function”.
- `jax.jacfwd`: forward-mode Jacobian
- `jax.jacrev`: reverse-mode Jacobian

Remember, Rule of thumb:
- forward-mode is good when input dimension is small
- reverse-mode is good when output dimension is small

In [ ]:
def f_vec(x):
    return jnp.array([x**2, jnp.sin(x), jnp.exp(x)])

print("jacfwd at 2.0:", jax.jacfwd(f_vec)(2.0))
print("jacrev at 2.0:", jax.jacrev(f_vec)(2.0))

In [ ]:
def F_vec(theta):
    # theta in R^2 -> output in R^3
    a, b = theta
    return jnp.array([a + b, a*b, a**2 + jnp.sin(b)])

theta = jnp.array([2.0, 0.5])

J_fwd = jax.jacfwd(F_vec)(theta)
J_rev = jax.jacrev(F_vec)(theta)

print("J_fwd:\n", J_fwd)
print("J_rev:\n", J_rev)

## Hessians

A Hessian is the matrix of second partial derivatives of a scalar function.
JAX can compute it directly.


In [ ]:
# Define a mean squared error loss function
def mse_loss(theta, x, y):
    yhat = theta[0] * x + theta[1]
    return jnp.mean((yhat - y)**2)

# Create some data
x = jnp.linspace(0.0, 1.0, 100)
y = 2.0 * x + 0.1

# Create an array of parameters (slope and intercept)
theta = jnp.array([1.0, 0.0])

# Compute the Hessian of the loss function
H = jax.hessian(mse_loss)(theta, x, y)

print("Hessian shape:", H.shape)
print(H)

Hp = jax.jacfwd(jax.grad(mse_loss))(theta, x, y)
print("Hessian via jacfwd/grad shape:", Hp.shape)
print(Hp)

Hpp = jax.jacrev(jax.jacfwd(mse_loss))(theta, x, y)
print("Hessian via jacfwd/jacfwd shape:", Hpp.shape)
print(Hpp)


## JAX JIT Compilation

JIT stands for Just-In-Time compilation.
In JAX, jax.jit compiles a Python function into optimized machine code via XLA.

Why this matters:
- Python is slow for tight numerical loops.
- JAX can fuse operations, optimize memory movement, and target CPU/GPU/TPU efficiently.
- Once compiled, repeated calls with the same input shapes/dtypes can be much faster.

Mental model:
- You write a function that looks like NumPy (using `jax.numpy`).
- jit turns it into an optimized kernel.
- The first call compiles; subsequent calls reuse the compiled executable.

In [ ]:
from datetime import datetime
import jax
import jax.numpy as jnp

def f_heavy(x):
    # A toy function that performs 50 repeated nonlinear operations
    for _ in range(50):
        x = jnp.sin(x) + 0.1 * x**2 - 0.05 * jnp.cos(2.0 * x)
    return jnp.mean(x)

# define an input array with 2,000,000 elements
x = jnp.linspace(-3.0, 3.0, 2000000)

# Eager timing
t0 = datetime.now()
y = f_heavy(x)
t1 = datetime.now()
print("eager result:", y)
print("eager time  :", t1 - t0, "s")


In [ ]:
f_heavy_jit = jax.jit(f_heavy)

# First call includes compilation time
t0 = datetime.now()
y = f_heavy_jit(x)
t1 = datetime.now()
print("jit (compile+run) result:", y)
print("jit (compile+run) time  :", t1 - t0, "s")


In [ ]:
# Second call is just execution time
t0 = datetime.now()
y = f_heavy_jit(x)
t1 = datetime.now()
print("jit (run) result:", y)
print("jit (run) time  :", t1 - t0, "s")

You can also use JIT as a decorator:

In [ ]:
@jax.jit
def f_heavy_jitdec(x):
    # A toy function that performs 50 repeated nonlinear operations
    for _ in range(50):
        x = jnp.sin(x) + 0.1 * x**2 - 0.05 * jnp.cos(2.0 * x)
    return jnp.mean(x)

# define an input array with 2,000,000 elements
x = jnp.linspace(-3.0, 3.0, 2000000)

# First call includes compilation time
t0 = datetime.now()
y = f_heavy_jitdec(x)
t1 = datetime.now()
print("jit_dec (compile+run) result:", y)
print("jit_dec (compile+run) time  :", t1 - t0, "s")

# Second call is just execution time
t0 = datetime.now()
y = f_heavy_jitdec(x)
t1 = datetime.now()
print("jit_dec (run) result:", y)
print("jit_dec (run) time  :", t1 - t0, "s")


## JIT + AD

JIT and AD compose seamlessly in JAX.
You can jit a function that uses grad, or grad a function that is jitted.

In [ ]:
# Define our mean squared error loss function
def mse_loss(theta, x, y):
    yhat = theta[0] * x + theta[1]
    return jnp.mean((yhat - y)**2)

# Create some data
x_in = jnp.linspace(0.0, 1.0, 50000)
y_in = 2.0 * x_in + 0.1
theta = jnp.array([1.0, 0.0])

# Eager evaluation of loss and gradient
t0 = datetime.now()
val, grad = jax.value_and_grad(mse_loss)(theta, x_in, y_in)
t1 = datetime.now()
print("eager loss  :", val)
print("eager grad  :", grad)
print("eager time  :", t1 - t0, "s")

# JIT-compiled evaluation of loss and gradient
loss_and_grad_jit = jax.jit(jax.value_and_grad(mse_loss))

# First call includes compilation time
val_jit, grad_jit = loss_and_grad_jit(theta, x_in, y_in)

# Second call is just execution time
t0 = datetime.now()
val_jit, grad_jit = loss_and_grad_jit(theta, x_in, y_in)
t1 = datetime.now()
print("jit loss    :", val_jit)
print("jit grad    :", grad_jit)
print("jit time    :", t1 - t0, "s")


## JIT Gotchas

JAX caches compiled code keyed by (function, shapes, dtypes, static args).

### Input Shapes and Dtypes Causes recompilation

If you call a jitted function with different shapes or dtypes, JAX will often compile a new version.

This is not “bad”, but you want to understand it to avoid surprise slowdowns and memory leaks.


In [ ]:
# Define a simple function
def f(x):
    jnp.sum(jnp.sin(x))

# Define inputs with different shapes
x1 = jnp.ones((1000,))
x2 = jnp.ones((2000,))  # different shape => likely will trigger another compile

# JIT the function
f_jit = jax.jit(f)

# Call with different shapes
out1 = f_jit(x1)
out2 = f_jit(x2)
print("Ran with two different shapes (may compile twice).")

### Static Arguments

If your function has arguments that affect control flow (e.g., number of steps in a loop), you should mark them as static using the `static_argnums` parameter in `jax.jit`. This tells JAX to treat these arguments as compile-time constants, preventing unnecessary recompilations for different values of these arguments.

In [ ]:
def f_static(x, n_steps):
    for _ in range(n_steps):  # Python loop: JAX needs n_steps at trace time
        x = jnp.sin(x) + 0.1 * x**2 - 0.05 * jnp.cos(2.0 * x)
    return jnp.mean(x)

# define input array
x = jnp.linspace(-3.0, 3.0, 200000)

# JIT the function without static_argnames
f_static_jit_broken = jax.jit(f_static)

# Call the function, passing n_steps as a regular argument
# This will fail to JIT properly if n_steps changes
y = f_static_jit_broken(x, n_steps=50)


In [ ]:
# JIT the function with static_argnames defined
f_static_jit = jax.jit(f_static, static_argnames=("n_steps",))

# Call the function, passing n_steps as a regular argument
# This should work correctly now
y = f_static_jit(x, n_steps=50)

print("result:", y)


### Print Statements

Normal Python `print()` usually won’t behave as expected inside jit functions,
because the function is being traced/compiled.

Use `jax.debug.print` for runtime printing from compiled code.


In [ ]:
from jax import debug

@jax.jit
def f_with_print(x):
    debug.print("Inside jit: mean(x) = {}", jnp.mean(x))
    return jnp.sum(jnp.sin(x))

_ = f_with_print(jnp.ones((10,))).block_until_ready()

### Control Flow

Python `if-statements` depend on concrete values and can cause tracing issues under `jit`.

If control flow depends on array values, use JAX control flow primitives:
- `jax.lax.cond` (if/else)
- `jax.lax.scan` (loops)

For today we’ll keep control flow simple, but it’s good to know these exist.

## JIT Summary

When to use `jit`:
- Inner loops / repeated evaluations
- Loss + gradient (`grad`) functions (optimization, inference)
- In concert with `vmap` for batched computations (see next section)

What to watch out for:
- Shape/dtype changes cause recompilation
- Python side effects don’t behave normally
- Value-dependent control flow needs `jax.lax.*` primitives

Big payoff:

- `jit` turns “NumPy-like Python” into fast compiled code.

## Using JAX `vmap`, automatic vectorization

`vmap` takes a function written for one example and transforms it to work on a batch of examples by automatically adding a mapped axis.

Why you should care and use `vmap`:
- Removes slow Python for loops
- Produces code that works cleanly with `jit` and `grad`
- Often enables better device utilization (e.g., vectorization on GPU/TPU)


In [ ]:
import jax
import jax.numpy as jnp
from datetime import datetime

def fsing(x):
    # x: shape (D,)
    return x / (jnp.linalg.norm(x) + 1e-8)

X = jnp.arange(120.0).reshape(4, 30)  # batch of 4 vectors in R^30
print("X shape:", X.shape)

In [ ]:
# Use the most basic way: Python for-loop
t0 = datetime.now()
outloop = []
for i in range(X.shape[0]):
    outloop.append(fsing(X[i]))
outloop = jnp.stack(outloop)
print("outloop shape:", outloop.shape)
print("Runtime: ", datetime.now() - t0, "s")

In [ ]:
# Use a more pythonic way using list comprehension
t0 = datetime.now()
outlist = jnp.stack([fsing(X[i]) for i in range(X.shape[0])])
print("outlist shape:", outlist.shape)
print("Runtime: ", datetime.now() - t0, "s")

In [ ]:
# Using JAX vmap
t0 = datetime.now()
fvmap = jax.vmap(fsing, in_axes=0, out_axes=0)
outvmap = fvmap(X)
print("outvmap shape:", outvmap.shape)
print("Runtime: ", datetime.now() - t0, "s")

In [ ]:
# Using JAX vmap + JIT
fjit = jax.jit(fvmap)

t0 = datetime.now()
outjit = fjit(X)
print("outjit shape:", outjit.shape)
print("Runtime: ", datetime.now() - t0, "s")

In [ ]:
# Can take gradients through jitted vmapped functions
fjitgrad = jax.jit(jax.vmap(jax.jacfwd(fsing), in_axes=0, out_axes=0))

t0 = datetime.now()
outgradjit = fjitgrad(X)
print("outgradjit shape:", outgradjit.shape)
print("Runtime: ", datetime.now() - t0, "s")

`in_axes` and `out_axes` specify which axes to map over for inputs and outputs, respectively. By default, `in_axes=0` and `out_axes=0`, meaning the first axis of the inputs and outputs are mapped over. You can customize these parameters to control how the function is vectorized.

In [ ]:
# Define a vectorized loss function using vmap
def loss_single(theta, x, y):
    # theta: (2,), x: scalar, y: scalar
    yhat = theta[0] * x + theta[1]
    return (yhat - y)**2

# Define some data
theta = jnp.array([2.0, -1.0])
xarr = jnp.linspace(-1.0, 1.0, 8)
yarr = 3.0 * xarr + 0.2

# transform loss_single into a batched version using vmap
# in_axes specifies which axes to map over for each input argument
loss_batch = jax.vmap(loss_single, in_axes=(None, 0, 0))  # theta shared, x/y batched
losses = loss_batch(theta, xarr, yarr)

`vmap` in one sentence:
It automatically batches a function by mapping one or more arguments over an axis.

Common patterns:
- `vmap(f, in_axes=(None, 0, 0))` → shared params, batched data
- `vmap(grad/jacfwd/jacrev(f))` → per-example gradients
- `jit(vmap(...))` → fast, batched, compiled execution

# Optax Optimization Library

Optax is a lightweight optimization library built for JAX.

In JAX/Optax, the training pattern is:
- Define a pure loss function
- Compute gradients with `value_and_grad`
- Use an Optax optimizer to produce parameter updates
- Apply updates to parameters

We’ll fit a quadratic model

 y = a x^2 + b x + c

to noisy samples.

In [ ]:
import jax
import jax.numpy as jnp
import optax

import matplotlib.pyplot as plt

# Generate synthetic data

# deterministic random key
key = jax.random.PRNGKey(0)

# Ground-truth parabola
true_params = {"a": 1.5, "b": -0.8, "c": 0.2}

# Sample x values and noisy y
N = 200
x = jnp.linspace(-2.0, 2.0, N)

key, subkey = jax.random.split(key)
noise_sigma = 0.1
y = (true_params["a"] * x**2 + true_params["b"] * x + true_params["c"] 
     + noise_sigma * jax.random.normal(subkey, shape=(N,)))
yerr = noise_sigma * jnp.ones_like(y)

Our “model” is just a function. Parameters live in a small vector `theta = [a, b, c]`.

In [ ]:
# Define our model
def model(theta, x):
    a, b, c = theta
    return a * x**2 + b * x + c

# Define mean squared error loss
def mse_loss(theta, x, y):
    yhat = model(theta, x)
    return jnp.mean((yhat - y) ** 2)

# Define a single training step, make it JIT-compiled for speed
@jax.jit
def train_step(theta, opt_state, x, y):
    # compute loss and gradients
    loss_val, grads = jax.value_and_grad(mse_loss)(theta, x, y)

    # update optimizer with gradients
    updates, opt_state = optimizer.update(grads, opt_state, params=theta)
    
    # apply updates to parameters and update optimizer state
    theta = optax.apply_updates(theta, updates)
    
    # return updated parameters, optimizer state, and loss value
    return theta, opt_state, loss_val

# Initialize parameters (intentionally not great)
theta0 = jnp.array([0.0, 0.0, 0.0])  # [a, b, c]

# setup SGD optimizer with learning rate = 1E-3
lr = 1E-3
optimizer = optax.sgd(learning_rate=lr)

# initialize optimizer state with initial values
opt_state = optimizer.init(theta0)

# Training loop for 5000 steps
num_steps = 5000
loss_history = []

# start training/fitting
for step in range(num_steps):
    # current parameters, if first step use initial parameters
    if step == 0:
        theta = theta0
    
    # perform a single training step
    theta, opt_state, loss_val = train_step(theta, opt_state, x, y)

    # record loss value for step
    loss_history.append(loss_val)

    # print progress every 500 steps
    if step % 500 == 0:
        print(f"step {step:4d} | loss = {loss_val:.4f} | theta = {theta}")
        
# Final parameters after training
theta_final = theta

print("Final fitted parameters:", theta_final)

In [ ]:
# Generate best-fit curve
y_fit = model(theta_final, x)

fig,axlist = plt.subplots(1,2,figsize=(12,5))
axlist[0].scatter(x, y, label="data", s=10)
axlist[0].plot(x, y_fit, color='orange', label="fit", linewidth=2)
axlist[0].set_xlabel("x")
axlist[0].set_ylabel("y")
axlist[0].legend()

axlist[1].plot(loss_history)
axlist[1].set_yscale('log')
axlist[1].set_xlabel("Training step")
axlist[1].set_ylabel("Loss (MSE)")


We can approximate the error on theta using the Hessian of the loss function. If we assume Gaussian errors, the covariance matrix of the best-fit parameters is the inverse of the Hessian of the loss at the optimum.

In [ ]:
# Negative log-likelihood function assuming Gaussian errors
def nll(theta, x, y, yerr):
    r = y - model(theta, x)
    return 0.5 * jnp.sum(r**2 / yerr**2)

# compute Hessian of NLL at best-fit parameters
H_nll = jax.hessian(nll)(theta_final, x, y, yerr)

# Invert Hessian -> covariance
cov = jnp.linalg.inv(H_nll)

# 1-sigma standard errors are sqrt(diagonal)
se = jnp.sqrt(jnp.diag(cov))

param_names = ["a", "b", "c"]
for name, val, err in zip(param_names, theta_final, se):
    print(f"{name} = {val:.6f} ± {err:.6f}")

# Probabilistic Programming with NumPyro

With Optax we found a single best-fit `theta`.
Now we’ll do Bayesian inference and compute the posterior distribution using NUTS (No-U-Turn Sampler), an adaptive Hamiltonian Monte Carlo algorithm.

This gives:
- credible intervals on parameters
- correlations / degeneracies
- posterior predictive uncertainty

Here I am only giving a brief overview of a simple toy example in NumPyro. For more details, see the [NumPyro documentation](https://num.pyro.ai/en/stable/).

In [ ]:
import jax
import jax.numpy as jnp

import numpyro
import numpyro.distributions as dist
from numpyro.infer import NUTS, MCMC, Predictive

import matplotlib.pyplot as plt

# Define our probabilistic model
def model(x, y=None, yerr=None):
    
    # Prior distributions on parameters, sampled from distributions
    a = numpyro.sample("a", dist.Normal(0.0, 5.0))
    b = numpyro.sample("b", dist.Normal(0.0, 5.0))
    c = numpyro.sample("c", dist.Normal(0.0, 5.0))

    # define our actual model
    mu = a * x**2 + b * x + c
    
    # Likelihood (sampling distribution) of observations
    numpyro.sample("y", dist.Normal(mu, yerr), obs=y)
    
# deterministic random key
key = jax.random.PRNGKey(0)

# Ground-truth parabola
true_params = {"a": 1.5, "b": -0.8, "c": 0.2}

# Sample x values and noisy y
N = 200
x = jnp.linspace(-2.0, 2.0, N)

key, subkey = jax.random.split(key)
noise_sigma = 0.1
y = (true_params["a"] * x**2 + true_params["b"] * x + true_params["c"] 
     + noise_sigma * jax.random.normal(subkey, shape=(N,)))
yerr = noise_sigma * jnp.ones_like(y)

# Set up NUTS MCMC
kernel = NUTS(model)
mcmc = MCMC(kernel, num_warmup=1000, num_samples=2000, num_chains=1, progress_bar=True)

# Run MCMC to sample from posterior
rng_key = jax.random.PRNGKey(1)
mcmc.run(rng_key, x=x, y=y, yerr=yerr)

# Print summary of MCMC results
mcmc.print_summary()

Now we want full posteriors on parameters:

In [ ]:
samples = mcmc.get_samples()
{k: v.shape for k, v in samples.items()}

Now let's plot our full joint distribution of parameters using a corner plot:

In [ ]:
import arviz as az
import matplotlib.pyplot as plt

# Convert NumPyro MCMC object -> ArviZ InferenceData
idata = az.from_numpyro(mcmc)

# Choose which parameters to include
var_names = ["a", "b", "c"]

# "Proper" corner plot
az.plot_pair(
    idata,
    var_names=var_names,
    kind="kde",          # smooth 2D densities
    marginals=True,      # 1D marginals on diagonal
    figsize=(10, 10),
)
plt.suptitle("Posterior corner plot (NumPyro NUTS)", y=1.02)
plt.show()

## For more information about NumPyro, see Dan Foreman-Mackey's tutorial: https://dfm.io/posts/intro-to-numpyro/

# Appendix: Training a Simple MLP with Flax NNX

Not covered in the lecture. This appendix shows how JAX concepts extend naturally to neural network training using Flax NNX, the modern Flax API.


In [ ]:
import jax
import jax.numpy as jnp
import optax
import flax.nnx as nnx
from functools import partial
import matplotlib.pyplot as plt

# -------------------------
# Model
# -------------------------
class MLP(nnx.Module):
    def __init__(self, *, hidden_dim=64, rngs: nnx.Rngs):
        self.dense1 = nnx.Linear(1, hidden_dim, rngs=rngs)
        self.dense2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs)
        self.out    = nnx.Linear(hidden_dim, 1, rngs=rngs)

    def __call__(self, x):
        x = nnx.relu(self.dense1(x))
        x = nnx.relu(self.dense2(x))
        return self.out(x)

# -------------------------
# Data
# -------------------------
key = jax.random.PRNGKey(0)
key, subkey = jax.random.split(key)
x = jnp.linspace(-2.0, 2.0, 200)[:, None]
y = 1.5 * x**2 - 0.8 * x + 0.2 + jax.random.normal(subkey, x.shape) * 0.1

# -------------------------
# Initialize model, then split into (graphdef, params/state)
# -------------------------
mlp = MLP(hidden_dim=64, rngs=nnx.Rngs(0))
model, params = nnx.split(mlp)   # graphdef is static; params are arrays

# -------------------------
# Optimizer state initialized from params
# -------------------------
optimizer = optax.adam(learning_rate=1E-3)
opt_state = optimizer.init(params)

# -------------------------
# Loss defined in terms of params + static model
# -------------------------
@partial(jax.jit, static_argnames=("model",))
def loss_fn(params, x, y, *, model):
    m = nnx.merge(model, params)   # rebuild module with current params
    yhat = m(x)
    return jnp.mean((yhat - y) ** 2)

# -------------------------
# One training step: update params (arrays) only
# -------------------------
@partial(jax.jit, static_argnames=("model",))
def train_step(params, opt_state, x, y, *, model):
    loss, grads = jax.value_and_grad(loss_fn)(params, x, y, model=model)
    updates, opt_state = optimizer.update(grads, opt_state, params=params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# -------------------------
# Training loop
# -------------------------
loss_hist = []
for step in range(5000):
    params, opt_state, loss = train_step(params, opt_state, x, y, model=model)
    loss_hist.append(loss)
    if step % 500 == 0:
        print(f"step {step:4d} | loss = {loss:.6f}")

# Rebuild trained model for prediction
trained_model = nnx.merge(model, params)
y_pred = trained_model(x)

# Plot
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.scatter(x, y, s=15, label="true")
ax.plot(x, y_pred, linewidth=2, label="MLP fit")
ax.legend()
ax.set_title("Flax NNX MLP fit (appendix)")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()